# Temporal Bibliometric Topic Modeling untuk Mendeteksi Evolusi dan Pergeseran Tren Riset Data Science

Berdasarkan framework **Temporal LDA**, notebook ini akan mengekstraksi topik laten dari dokumen menggunakan N-gram & TF-IDF, mencari jumlah topik optimal (K) melalui kurva Coherence vs Perplexity, dan memetakan evolusi topik secara dinamis per tahun (Jalur Migrasi Riset).

In [ ]:
# Install pustaka yang dibutuhkan jika belum ada
!pip install -q pyLDAvis gensim nltk pandas matplotlib seaborn wordcloud ipywidgets

In [ ]:
import pandas as pd
import numpy as np
import re
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=FutureWarning)

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

import gensim
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel, TfidfModel
from gensim.models.phrases import Phrases, Phraser

import pyLDAvis
import pyLDAvis.gensim_models as gensimvis

import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud
import ipywidgets as widgets
from IPython.display import display, clear_output

# Download NLTK data
nltk.download('punkt', quiet=True)
nltk.download('stopwords', quiet=True)

## 1. Pengumpulan Data & Filtering Predatory Journals

In [ ]:
# Load Dataset (Harap sesuaikan path dengan direktori Anda)
# Untuk Google Colab: df = pd.read_csv('/content/dataset_riset_Data Science-Semantic Scholer.csv')
file_path = 'dataset_riset_Data Science-Semantic Scholer.csv' 
try:
    df = pd.read_csv(file_path)
    print(f"Berhasil memuat dataset dengan {len(df)} baris.")
except FileNotFoundError:
    print(f"File {file_path} tidak ditemukan. Pastikan file berada di direktori yang sama.")
    # Fallback dummy data jika tidak ada
    # df = pd.DataFrame({'Year': [2020]*10, 'Source': ['']*10, 'Publisher': ['']*10, 'Abstract': ['data science']*10})
    raise

# Filter Predatory Journals
predatory_keywords = [
    'omics', 'waset', 'academic journals', 'sciedu',
    'david publishing', 'baishideng', 'bentham open',
    'iosr', 'science domains', 'lap lambert'
]

df['Publisher_lower'] = df['Publisher'].astype(str).str.lower().fillna('')
df['Source_lower'] = df['Source'].astype(str).str.lower().fillna('')

def is_predatory(row):
    for keyword in predatory_keywords:
        if keyword in row['Publisher_lower'] or keyword in row['Source_lower']:
            return True
    return False

df['is_predatory'] = df.apply(is_predatory, axis=1)
num_predatory = df['is_predatory'].sum()
print(f"Ditemukan {num_predatory} artikel terindikasi predator.")

# Bersihkan dataset
df = df[~df['is_predatory']].copy()
df = df.drop(columns=['Publisher_lower', 'Source_lower', 'is_predatory'])
print(f"Sisa dokumen bersih: {len(df)}")
# Buang baris dengan Abstract kosong
df = df.dropna(subset=['Abstract'])
print(f"Dokumen setelah hapus abstrak kosong: {len(df)}")

## 2. NLP Pipeline & Inovasi Ekstraksi (Phrase ID + TF-IDF)

In [ ]:
# Cleaning & Tokenisasi Dasar
stop_words = set(stopwords.words('english'))
custom_stops = {'research', 'paper', 'method', 'using', 'result', 'data', 'science', 'model', 'approach', 'study', 'show', 'propose'}
stop_words.update(custom_stops)

def preprocess_basic(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    tokens = word_tokenize(text)
    tokens = [word for word in tokens if word not in stop_words and len(word) > 2]
    return tokens

print("Melakukan tokenisasi dan pembersihan dasar...")
docs = df['Abstract'].apply(preprocess_basic).tolist()

# Phrase Identification (Bigrams / Trigrams)
print("Membangun model Bigram untuk identifikasi frasa...")
bigram = Phrases(docs, min_count=3, threshold=10)
bigram_mod = Phraser(bigram)

# Terapkan bigram ke dokumen
clean_docs = [bigram_mod[doc] for doc in docs]
df['clean_tokens'] = clean_docs

print("Contoh dokumen setelah Bigram:")
print(clean_docs[0][:10])

In [ ]:
# Membangun Dictionary
dictionary = corpora.Dictionary(clean_docs)
# Filter ekstrim: kata harus muncul minimal di 2 dokumen, maksimal di 50% dokumen
dictionary.filter_extremes(no_below=2, no_above=0.5)

# Membuat format Bag-of-Words (Term Frequency)
corpus_tf = [dictionary.doc2bow(text) for text in clean_docs]

# Inovasi Ekstraksi: Menggunakan pembobotan TF-IDF
print("Menerapkan pembobotan TF-IDF...")
tfidf_model = TfidfModel(corpus_tf)
corpus_tfidf = tfidf_model[corpus_tf]

print(f"Ukuran kamus (V): {len(dictionary)}")
print(f"Jumlah dokumen (N): {len(corpus_tfidf)}")

## 3. Kalibrasi Model (Mencari Titik Puncak Semantik K)

In [ ]:
# Perhatian: Bagian ini memakan waktu komputasi yang lumayan. 
# Rentang K yang dicari: 2 hingga 20 (step 2)
limit = 22; start = 2; step = 2

coherence_values = []
perplexity_values = []
model_list = []

print("Mulai melatih beberapa model untuk mencari jumlah topik (K) optimal...")
for num_topics in range(start, limit, step):
    # Pelatihan Model LDA menggunakan Corpus TF-IDF
    model = LdaModel(corpus=corpus_tfidf, id2word=dictionary, num_topics=num_topics, 
                     random_state=42, passes=10, iterations=100, alpha='auto')
    model_list.append(model)
    
    # Hitung Coherence (Cv)
    coherencemodel = CoherenceModel(model=model, texts=clean_docs, dictionary=dictionary, coherence='c_v')
    coherence_values.append(coherencemodel.get_coherence())
    
    # Hitung Perplexity
    perplexity = model.log_perplexity(corpus_tfidf)
    perplexity_values.append(perplexity)
    
    print(f"K = {num_topics} | Coherence: {coherence_values[-1]:.4f} | Perplexity: {perplexity_values[-1]:.4f}")

# Visualisasi Kalibrasi (Sweet Spot)
x = range(start, limit, step)

fig, ax1 = plt.subplots(figsize=(10, 6))

color = 'tab:blue'
ax1.set_xlabel('Jumlah Topik (K)')
ax1.set_ylabel('Skor Koherensi (Cv)', color=color)
ax1.plot(x, coherence_values, marker='o', color=color, linewidth=2, label='Coherence (Cv)')
ax1.tick_params(axis='y', labelcolor=color)

# Axis kedua untuk Perplexity
ax2 = ax1.twinx()  
color = 'tab:orange'
ax2.set_ylabel('Perplexity', color=color)  
ax2.plot(x, perplexity_values, marker='x', linestyle='--', color=color, linewidth=2, label='Perplexity')
ax2.tick_params(axis='y', labelcolor=color)

# Garis vertikal di titik koherensi tertinggi
optimal_idx = np.argmax(coherence_values)
optimal_k = x[optimal_idx]
plt.axvline(x=optimal_k, color='red', linestyle=':', label=f'Sweet Spot (K={optimal_k})')

fig.tight_layout()  
plt.title('Kalibrasi Model: Mencari Puncak Semantik (Coherence vs Perplexity)')
plt.grid(True, alpha=0.3)
fig.legend(loc="upper right", bbox_to_anchor=(1,1), bbox_transform=ax1.transAxes)
plt.show()

## 4. Pelatihan Model Optimal

In [ ]:
# Memilih model dengan Skor Koherensi (Cv) tertinggi
print(f"Topik optimal terpilih adalah K = {optimal_k}")
best_lda_model = model_list[optimal_idx]

print("\n=== TOPIK DOMINAN ===")
for idx, topic in best_lda_model.print_topics(-1):
    print(f"Topik {idx}: {topic}\n")

## 5. Temporal Unveiling (Analisis Jalur Migrasi Riset 2017-2026)

In [ ]:
def get_dominant_topic(bow, model):
    topic_probs = model.get_document_topics(bow)
    if not topic_probs:
        return None
    return max(topic_probs, key=lambda x: x[1])[0]

# Terapkan prediksi topik ke seluruh dokumen
df['dominant_topic'] = [get_dominant_topic(bow, best_lda_model) for bow in corpus_tfidf]

# Kelompokkan berdasarkan Tahun dan Topik
temporal_topics = df.groupby(['Year', 'dominant_topic']).size().unstack(fill_value=0)

# Filter tahun jika perlu (misal: 2017-2026)
temporal_topics = temporal_topics[(temporal_topics.index >= 2017) & (temporal_topics.index <= 2026)]

# Normalisasi menjadi persentase (Proporsi Ketertarikan %)
temporal_perc = temporal_topics.div(temporal_topics.sum(axis=1), axis=0) * 100

plt.figure(figsize=(12, 6))
for col in temporal_perc.columns:
    plt.plot(temporal_perc.index, temporal_perc[col], marker='o', linewidth=2, label=f'Topik {col}')

plt.title('Menyingkap Evolusi: Jalur Migrasi Riset', fontsize=16)
plt.xlabel('Tahun', fontsize=12)
plt.ylabel('Proporsi Ketertarikan (%)', fontsize=12)
plt.xticks(temporal_perc.index)
plt.grid(True, alpha=0.5)
plt.legend(title='ID Topik', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## 6. Dashboard Interaktif (Digital Topography & Visualisasi)

In [ ]:
import pyLDAvis.gensim_models as gensimvis

# 1. Persiapkan pyLDAvis
pyLDAvis.enable_notebook()
# pyLDAvis menggunakan corpus TF agar visualisasinya lebih tepat merepresentasikan Term Frequency
vis_data = gensimvis.prepare(best_lda_model, corpus_tf, dictionary)

# 2. Fungsi WordCloud
def plot_wordcloud(lda_model):
    topics = lda_model.show_topics(num_topics=optimal_k, num_words=15, formatted=False)
    cols = min(4, len(topics))
    rows = (len(topics) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5*rows), sharex=True, sharey=True)
    axes = np.array(axes).flatten()
    
    for i in range(len(axes)):
        if i < len(topics):
            topic_words = dict(topics[i][1])
            cloud = WordCloud(background_color='white', width=400, height=400,
                              colormap='viridis').generate_from_frequencies(topic_words)
            axes[i].imshow(cloud, interpolation='bilinear')
            axes[i].set_title(f'Topic {topics[i][0]}', fontdict=dict(size=16))
        axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# 3. Fungsi Bar Chart
def plot_barchart(lda_model):
    topics = lda_model.show_topics(num_topics=optimal_k, num_words=15, formatted=False)
    cols = min(4, len(topics))
    rows = (len(topics) + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(20, 5*rows), sharey=False)
    axes = np.array(axes).flatten()

    for i in range(len(axes)):
        if i < len(topics):
            words = [word for word, prob in topics[i][1]]
            probs = [prob for word, prob in topics[i][1]]
            axes[i].barh(words, probs, color='steelblue')
            axes[i].set_title(f'Topic {topics[i][0]}')
            axes[i].invert_yaxis()
        else:
            axes[i].axis('off')

    plt.tight_layout()
    plt.show()

# Buat Widget Output
out_temporal = widgets.Output()
out_pyldavis = widgets.Output()
out_wordcloud = widgets.Output()
out_barchart = widgets.Output()

with out_temporal:
    fig, ax = plt.subplots(figsize=(10, 5))
    sns.heatmap(temporal_perc, annot=True, cmap='YlGnBu', fmt='.1f', ax=ax)
    ax.set_title('Heatmap Evolusi Topik (Persentase)')
    plt.tight_layout()
    plt.show()

with out_pyldavis:
    display(vis_data)

with out_wordcloud:
    plot_wordcloud(best_lda_model)
    
with out_barchart:
    plot_barchart(best_lda_model)

# Susun Tab
tab_visuals = widgets.Tab(children=[out_temporal, out_pyldavis, out_wordcloud, out_barchart])
tab_visuals.set_title(0, 'Heatmap Temporal')
tab_visuals.set_title(1, 'Topografi Digital (pyLDAvis)')
tab_visuals.set_title(2, 'Word Clouds')
tab_visuals.set_title(3, 'Bar Charts')

# Final Dashboard Layout
header = widgets.HTML('''
    <div style="background-color: #f8f9fa; padding: 20px; border-radius: 10px; border-left: 5px solid #0056b3;">
        <h2 style="margin:0; color: #0056b3;">Dashboard Analisis Topik - Temporal LDA</h2>
        <p style="margin:5px 0 0 0;">Menampilkan topografi digital, evolusi semantik, dan kata kunci paling relevan (TF-IDF + Phrase ID).</p>
    </div>
''')

display(widgets.VBox([header, tab_visuals]))
